# 06 - Qdrant 向量搜索深度测试 (Vector Search Deep Dive)

**目的**: 精确测量 Qdrant 向量搜索延迟，找出影响搜索性能的关键参数  
**测量变量**: top_k / collection_size / payload_filter / score_threshold

**关键问题**:
- 当知识库增长到 1000/10000 条时，搜索延迟如何变化？
- top_k=5 vs top_k=20 差多少？
- 加了 payload filter 之后性能损失多大？
- 现有 collection 的实际延迟是多少？

## 0. 配置参数

In [ ]:
import os
from datetime import datetime

QDRANT_URL = "http://localhost:6333"

# 生产环境 Collection（已存在）
PROD_COLLECTION = os.getenv("QDRANT_COLLECTION", "xiaocai_knowledge")
VECTOR_DIM = 384  # all-MiniLM-L6-v2

# 测试 top_k 变量
TOP_K_VALUES = [1, 3, 5, 10, 20, 50]

# 基准测试用的临时 Collection（完全控制 collection_size）
BENCH_COLLECTION = "benchmark_test_collection"
# 要测试的 Collection 大小
COLLECTION_SIZES = [100, 500, 1000, 5000]

# 每组测试轮数
N_ITERATIONS = 20
# 是否创建临时 Collection 进行 collection_size 测试（会写入数据）
RUN_SIZE_BENCHMARK = True

DATA_DIR = "../data"
os.makedirs(DATA_DIR, exist_ok=True)
RESULTS_FILE = f"{DATA_DIR}/vector_search_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"

print(f"Qdrant URL: {QDRANT_URL}")
print(f"生产 Collection: {PROD_COLLECTION}")
print(f"向量维度: {VECTOR_DIM}")
print(f"top_k 测试值: {TOP_K_VALUES}")
print(f"size 基准测试: {RUN_SIZE_BENCHMARK}")

## 1. 依赖导入

In [ ]:
import httpx
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

print("依赖导入完成")

## 2. Qdrant 工具函数

In [ ]:
def random_vector(dim: int) -> list:
    v = [random.gauss(0, 1) for _ in range(dim)]
    norm = (sum(x**2 for x in v) ** 0.5) or 1
    return [x / norm for x in v]


def qdrant_search(url: str, collection: str, vector: list, top_k: int,
                  payload_filter: dict = None, score_threshold: float = None) -> dict:
    """执行向量搜索，返回延迟和结果数"""
    payload = {"vector": vector, "limit": top_k, "with_payload": True}
    if payload_filter:
        payload["filter"] = payload_filter
    if score_threshold is not None:
        payload["score_threshold"] = score_threshold

    start = time.perf_counter()
    try:
        resp = httpx.post(f"{url}/collections/{collection}/points/search",
                          json=payload, timeout=15.0)
        elapsed = (time.perf_counter() - start) * 1000
        if resp.status_code == 200:
            results = resp.json().get("result", [])
            return {"success": True, "latency_ms": elapsed, "results_count": len(results),
                    "top_score": results[0]["score"] if results else 0.0}
        return {"success": False, "latency_ms": elapsed, "error": f"HTTP {resp.status_code}: {resp.text[:100]}"}
    except Exception as e:
        return {"success": False, "latency_ms": (time.perf_counter() - start) * 1000, "error": str(e)}


def create_bench_collection(url: str, name: str, dim: int, n_points: int) -> dict:
    """创建临时测试 Collection 并写入 n_points 个随机向量"""
    # 删除已存在的
    httpx.delete(f"{url}/collections/{name}", timeout=10.0)
    time.sleep(0.5)

    # 创建
    create_resp = httpx.put(
        f"{url}/collections/{name}",
        json={"vectors": {"size": dim, "distance": "Cosine"}},
        timeout=10.0
    )
    if create_resp.status_code not in (200, 201):
        return {"success": False, "error": f"创建失败: {create_resp.text[:200]}"}

    # 批量写入（每批最多 100 条）
    batch_size = 100
    for start_idx in range(0, n_points, batch_size):
        end_idx = min(start_idx + batch_size, n_points)
        points = [
            {
                "id": i,
                "vector": random_vector(dim),
                "payload": {"category": f"cat_{i % 10}", "source": "benchmark"}
            }
            for i in range(start_idx, end_idx)
        ]
        upsert_resp = httpx.put(
            f"{url}/collections/{name}/points",
            json={"points": points},
            timeout=30.0
        )
        if upsert_resp.status_code not in (200, 201):
            return {"success": False, "error": f"写入失败 batch {start_idx}: {upsert_resp.text[:200]}"}

    return {"success": True, "points": n_points}


def delete_bench_collection(url: str, name: str):
    httpx.delete(f"{url}/collections/{name}", timeout=10.0)


print("工具函数加载完成")

## 3. 生产 Collection 现状测试

In [ ]:
print("=" * 60)
print(f"生产 Collection: {PROD_COLLECTION}")
print("=" * 60)

# 检查 collection 信息
info_resp = httpx.get(f"{QDRANT_URL}/collections/{PROD_COLLECTION}", timeout=5.0)
if info_resp.status_code == 200:
    info = info_resp.json().get("result", {})
    print(f"向量数:   {info.get('vectors_count', 'N/A')}")
    print(f"点数:     {info.get('points_count', 'N/A')}")
    print(f"状态:     {info.get('status', 'N/A')}")
    PROD_COLLECTION_EXISTS = True
else:
    print(f"Collection 不存在或不可达 (HTTP {info_resp.status_code})")
    PROD_COLLECTION_EXISTS = False

# 测试生产 Collection 的搜索延迟（top_k=5）
prod_results = []
if PROD_COLLECTION_EXISTS:
    print(f"\n测试搜索延迟 ({N_ITERATIONS} 轮, top_k=5):")
    for i in range(N_ITERATIONS):
        vec = random_vector(VECTOR_DIM)
        r = qdrant_search(QDRANT_URL, PROD_COLLECTION, vec, top_k=5)
        prod_results.append({"iteration": i+1, "latency_ms": r["latency_ms"], "success": r["success"],
                              "results_count": r.get("results_count", 0)})

    df_prod = pd.DataFrame(prod_results)
    df_prod_ok = df_prod[df_prod["success"]]
    if not df_prod_ok.empty:
        print(f"  P50: {np.percentile(df_prod_ok['latency_ms'], 50):.2f} ms")
        print(f"  P95: {np.percentile(df_prod_ok['latency_ms'], 95):.2f} ms")
        print(f"  Max: {df_prod_ok['latency_ms'].max():.2f} ms")
        print(f"  平均返回结果数: {df_prod_ok['results_count'].mean():.1f}")
else:
    df_prod = pd.DataFrame()
    df_prod_ok = pd.DataFrame()
    print("跳过生产 Collection 测试")

## 4. top_k 参数对搜索延迟的影响

In [ ]:
print(f"top_k 延迟测试 (使用 {PROD_COLLECTION if PROD_COLLECTION_EXISTS else BENCH_COLLECTION})")
print("=" * 60)

# 如果生产 Collection 不存在，创建一个 1000 条的临时 Collection
if not PROD_COLLECTION_EXISTS:
    print("创建临时 Collection (1000 条向量)...")
    r = create_bench_collection(QDRANT_URL, BENCH_COLLECTION, VECTOR_DIM, 1000)
    if r["success"]:
        SEARCH_COLLECTION = BENCH_COLLECTION
        PROD_COLLECTION_EXISTS = True
        print(f"临时 Collection 创建成功: {BENCH_COLLECTION}")
    else:
        print(f"创建失败: {r.get('error')}")
        SEARCH_COLLECTION = None
else:
    SEARCH_COLLECTION = PROD_COLLECTION

topk_results = []
if SEARCH_COLLECTION:
    for k in TOP_K_VALUES:
        for i in range(N_ITERATIONS):
            vec = random_vector(VECTOR_DIM)
            r = qdrant_search(QDRANT_URL, SEARCH_COLLECTION, vec, top_k=k)
            topk_results.append({"top_k": k, "iteration": i+1, "latency_ms": r["latency_ms"], "success": r["success"]})

    df_topk = pd.DataFrame(topk_results)
    df_topk_ok = df_topk[df_topk["success"]]

    print("\ntop_k × 延迟统计 (ms):")
    topk_stats = df_topk_ok.groupby("top_k")["latency_ms"].agg(
        P50=lambda x: np.percentile(x, 50),
        P95=lambda x: np.percentile(x, 95),
        Max="max",
        Mean="mean"
    ).round(2)
    print(topk_stats.to_string())
else:
    df_topk = pd.DataFrame()
    df_topk_ok = pd.DataFrame()
    print("无可用 Collection，跳过测试")

## 5. Collection 大小 × 搜索延迟（扩展性基准）

In [ ]:
size_results = []

if RUN_SIZE_BENCHMARK:
    print(f"Collection 大小 × 搜索延迟 (top_k=5)")
    print("=" * 60)
    print("注意: 此测试会创建临时 Collection 并写入大量随机向量")

    for n_points in COLLECTION_SIZES:
        print(f"\n正在创建 {n_points} 条向量的 Collection...")
        r = create_bench_collection(QDRANT_URL, BENCH_COLLECTION, VECTOR_DIM, n_points)

        if not r["success"]:
            print(f"  创建失败: {r.get('error')}")
            continue

        # 等待索引建立
        time.sleep(1)

        times = []
        for _ in range(N_ITERATIONS):
            vec = random_vector(VECTOR_DIM)
            r = qdrant_search(QDRANT_URL, BENCH_COLLECTION, vec, top_k=5)
            if r["success"]:
                times.append(r["latency_ms"])
                size_results.append({"collection_size": n_points, "latency_ms": r["latency_ms"], "success": True})

        if times:
            print(f"  {n_points:6d} 条 | P50={np.percentile(times,50):.2f}ms | P95={np.percentile(times,95):.2f}ms | Max={max(times):.2f}ms")

    # 清理临时 Collection
    delete_bench_collection(QDRANT_URL, BENCH_COLLECTION)
    print(f"\n临时 Collection '{BENCH_COLLECTION}' 已清理")

    df_size = pd.DataFrame(size_results)
else:
    df_size = pd.DataFrame()
    print("RUN_SIZE_BENCHMARK=False，跳过扩展性测试")

## 6. Payload Filter 性能影响

In [ ]:
print("Payload Filter 性能影响测试")
print("=" * 60)

if not SEARCH_COLLECTION:
    print("无可用 Collection，跳过")
else:
    filter_results = []

    filter_scenarios = [
        ("无 filter",  None),
        ("match filter", {"must": [{"key": "category", "match": {"value": "办公设备"}}]}),
        ("range filter", {"must": [{"key": "score", "range": {"gte": 0.5}}]}),
    ]

    for scenario_name, filter_payload in filter_scenarios:
        for i in range(N_ITERATIONS):
            vec = random_vector(VECTOR_DIM)
            r = qdrant_search(QDRANT_URL, SEARCH_COLLECTION, vec, top_k=5,
                              payload_filter=filter_payload)
            filter_results.append({
                "scenario": scenario_name,
                "iteration": i + 1,
                "latency_ms": r["latency_ms"],
                "success": r["success"],
                "results_count": r.get("results_count", 0),
            })

    df_filter = pd.DataFrame(filter_results)
    df_filter_ok = df_filter[df_filter["success"]]

    print("\nFilter 场景延迟统计 (ms):")
    filter_stats = df_filter_ok.groupby("scenario")["latency_ms"].agg(
        P50=lambda x: np.percentile(x, 50),
        P95=lambda x: np.percentile(x, 95),
        Max="max",
        Mean="mean"
    ).round(2)
    print(filter_stats.to_string())
    print("\n注意: filter 对无匹配字段数据不会有显著影响（Qdrant 内置优化）")

## 7. 综合可视化

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Qdrant 向量搜索深度分析", fontsize=14)

# 图1: top_k vs P50 延迟
ax1 = axes[0]
if not df_topk.empty and not df_topk_ok.empty:
    topk_p50 = df_topk_ok.groupby("top_k")["latency_ms"].quantile(0.5)
    topk_p95 = df_topk_ok.groupby("top_k")["latency_ms"].quantile(0.95)
    ax1.plot(topk_p50.index, topk_p50.values, "bo-", label="P50", linewidth=2)
    ax1.plot(topk_p95.index, topk_p95.values, "r^--", label="P95", linewidth=2)
    ax1.set_xlabel("top_k")
    ax1.set_ylabel("延迟 (ms)")
    ax1.set_title("top_k × 搜索延迟")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
else:
    ax1.text(0.5, 0.5, "无数据", ha='center', va='center', transform=ax1.transAxes)
    ax1.set_title("top_k × 搜索延迟")

# 图2: Collection 大小 × 延迟
ax2 = axes[1]
if not df_size.empty:
    size_grouped = df_size[df_size["success"]].groupby("collection_size")["latency_ms"]
    sizes = list(size_grouped.groups.keys())
    p50_vals = [size_grouped.get_group(s).quantile(0.5) for s in sizes]
    p95_vals = [size_grouped.get_group(s).quantile(0.95) for s in sizes]
    ax2.plot(sizes, p50_vals, "bo-", label="P50", linewidth=2)
    ax2.plot(sizes, p95_vals, "r^--", label="P95", linewidth=2)
    ax2.set_xlabel("Collection 大小 (向量数)")
    ax2.set_ylabel("延迟 (ms)")
    ax2.set_title("Collection 大小 × 搜索延迟")
    ax2.legend()
    ax2.grid(True, alpha=0.3)
else:
    ax2.text(0.5, 0.5, "未运行\n(RUN_SIZE_BENCHMARK=False)", ha='center', va='center', transform=ax2.transAxes)
    ax2.set_title("Collection 大小 × 搜索延迟")

# 图3: Filter 影响对比
ax3 = axes[2]
if 'df_filter' in dir() and not df_filter.empty:
    scenarios = df_filter["scenario"].unique()
    data3 = [df_filter[df_filter["scenario"] == s]["latency_ms"].values for s in scenarios]
    ax3.boxplot(data3, labels=scenarios, patch_artist=True)
    ax3.set_ylabel("延迟 (ms)")
    ax3.set_title("Payload Filter 性能影响")
    ax3.tick_params(axis='x', rotation=15)
else:
    ax3.text(0.5, 0.5, "无数据", ha='center', va='center', transform=ax3.transAxes)
    ax3.set_title("Payload Filter 性能影响")

plt.tight_layout()
chart_file = f"../data/vector_search_chart_{datetime.now().strftime('%Y%m%d_%H%M')}.png"
plt.savefig(chart_file, dpi=150, bbox_inches="tight")
plt.show()
print(f"图表已保存: {chart_file}")

## 8. 保存结果

In [ ]:
all_results = []
if not df_topk.empty:
    df_topk["test_group"] = "topk"
    all_results.append(df_topk)
if not df_size.empty:
    df_size["test_group"] = "size"
    all_results.append(df_size)
if 'df_filter' in dir() and not df_filter.empty:
    df_filter["test_group"] = "filter"
    all_results.append(df_filter)

if all_results:
    pd.concat(all_results, ignore_index=True).to_csv(RESULTS_FILE, index=False)
    print(f"结果已保存: {RESULTS_FILE}")
else:
    print("无结果（Qdrant 不可达或 Collection 不存在）")